# Setup
## Importowanie Bibliotek

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

## Pomocne Funkcje

In [ ]:
def boxplot(df, column_name):
    """
    Displays a boxplot of the choosen column to show its distribution.
    Parameters:
        df (pandas.DataFrame): The DataFrame containing the data.
        column_name (str): The name of the categorical column to plot.
    Returns:
        None. Displays the bar chart.
    """
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[column_name].dropna(), color='lightgreen') # as you can see, you can use sns and plt at the same time
    plt.title(f'Distribution of {column_name} (boxplot)')
    plt.xlabel(column_name)

    plt.show()

def one_hot_encode_column(dataframe, column_name):
    return pd.get_dummies(dataframe, columns=[column_name], prefix=column_name)


def categorical_and_survived(df, categorical_column):
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x=categorical_column, y='Survived', palette='Set2')
    plt.title(f'Survived by {categorical_column}')
    plt.xlabel(categorical_column)
    plt.ylabel('Survived')
    plt.tight_layout()
    plt.show()

def plot_numeric_histogram(df, column_name):
    """
    Plots a histogram for the specified column in the DataFrame,
    and adds vertical lines for the mean and median.
    Parameters:
        df (pandas.DataFrame): The DataFrame containing the data.
        column_name (str): The name of the column to plot.
    Returns:
        None. Displays the histogram.
    """

    data = df[column_name].dropna()
    mean_val = data.mean()
    median_val = data.median()

    plt.figure(figsize=(6, 4))
    plt.hist(data, bins=30, color='steelblue', edgecolor='black')
    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    plt.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')

    plt.title(f'Distribution of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Ładowanie Danych

In [ ]:
base_path = Path("01_przetwarzanie_wizualizacja_danych") # change to path folder with data
titanic_df = pd.read_csv( base_path / 'titanic.csv', index_col='PassengerId')
titanic_df

# Analiza
## Istotność Kolumn
Moi kandydaci na ważne kolumny:
* Survived - to co predykujemy
* Pclass - bogacze vs biedni
* Sex - "Najpierw kobiety i dzieci"
* Age - witalność może mieć wpływ na przetrwanie
* SibSp / Parch - wpływ rozmiaru rodziny na przetrwanie
* Fare - aparat porównawczy wewnątrz danej Pclass (bogaci vs bogatsi)
* Cabin - numer kabiny (do ekstrakcji lokalizacji na pokładzie)
* Embarked - miejsce wejścia na pokład może korelować z innymi cechami pasażera

In [ ]:
titanic_df = titanic_df.drop(columns=['Name', 'Ticket'])
titanic_df.info()

## Ekstrakcja
Numer kabiny jako tako nie jest nam potrzebny, natomiast literka służąca jako prefix to lokalizacja na pokładzie gdzie znajdował się pokój, co może być interesujące w kontekście bliskości do raft ratunkowych.

*Cabin*:**C134** -> *Location*:**C**

In [ ]:
titanic_df['Location'] = titanic_df['Cabin'].str[0]
titanic_df = titanic_df.drop('Cabin', axis='columns')
titanic_df.info()


Wyodrębnienie SibSp i Parch jest trochę zbyt szczegółowe - chciałbym przeanalizować jakie szanse na przeżycie ma człowiek ze względu na dużą *rodzinę*

In [ ]:
titanic_df['Family'] = titanic_df['Parch'] + titanic_df['SibSp']
titanic_df = titanic_df.drop('Parch', axis='columns')
titanic_df = titanic_df.drop('SibSp', axis='columns')
titanic_df.info()

Czysto optymalizacyjnie, zamienię kolumnę 'Sex' na 'IsMale' z wartością typu bool:
* True -> memszczyzna
* False -> kombieta

In [ ]:
titanic_df['IsMale'] = titanic_df['Sex'] == 'male'
titanic_df = titanic_df.drop('Sex', axis='columns')
titanic_df.info()

## Braki w Wartościach

In [ ]:
titanic_df.isna().sum()

### Dystrybucja Wieku
Wiek:
* duży słupek najmłodszych
* dzwon wieku osób z centrum w ~25 lat
* długi ogon boomerów

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Turn this to function and use for fare
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

age = titanic_df['Age']
age_cleaned = age.dropna()

mean_age = age_cleaned.mean()
median_age = age_cleaned.median()

sns.histplot(age_cleaned, kde=True, ax=axes, color='skyblue')
axes.axvline(mean_age, color='red', linestyle='--', label=f'Mean ({mean_age:.1f})')
axes.axvline(median_age, color='green', linestyle='-', label=f'Median ({median_age:.1f})')
axes.set_title('Age Distribution')
axes.legend()

Podrążmy to trochę!
Uważam, iż `Age` może ładnie sie korelować z wartością `Pclass` albo `Fare`, gdyż wraz z wiekiem człowiek często jest bardziej majętny.
Dokonam analizy wieku przez pryzmat `Pclass`y

In [ ]:
print(titanic_df['Pclass'].unique())

In [ ]:
boxplot(titanic_df[titanic_df['Pclass'] == 0], "Age")

In [ ]:
boxplot(titanic_df[titanic_df['Pclass'] == 1], "Age")

In [ ]:
boxplot(titanic_df[titanic_df['Pclass'] == 2], "Age")

Mimo trochę overlappu, moja hipoteza się sprawdziła: `Age` rośnie tym wyższa klasa (niższa `Pclass`)
Uważam, iż najlepiej jest to załatwić medianą danej grupy `Pclass`:

In [ ]:
titanic_df['Age'] = titanic_df.groupby(['Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

### Dystrybucja Lokacji na Pokładzie
Mega dużo danych nam brakuje (77%!)

In [ ]:
titanic_df['Location'].isna().mean() * 100

Biorąc pod uwagę ten masywny brak danych, najlepszym rozwiązaniem jest utworzenie oddzielnej kategorii dla niezidentifikowanej lokalizacji pokoju

In [ ]:
registered_locations = titanic_df['Location'].unique()
unspecified_dummy_location = 'U'
if unspecified_dummy_location not in registered_locations:
  titanic_df['Location'] = titanic_df['Location'].fillna(unspecified_dummy_location)
titanic_df['Location'].unique()

In [ ]:
titanic_df['Location'].isna().sum()

### Dystrybucja Ceny Biletu (Fare)
Dystrybucje cen biletu (rozpatrzone wewnątrz danej Pclass'y) są asymetryczne (tak jak wiek)

In [ ]:
fig, axes = plt.subplots(1, len(titanic_df['Pclass'].unique()), figsize=(12, 4))
index = 0
for passenger_class in titanic_df['Pclass'].unique():
  fare = titanic_df[titanic_df['Pclass'] == passenger_class] # filter this shyt

  fare = fare['Fare']
  fare_cleaned = fare.dropna()

  mean_fare = fare_cleaned.mean()
  median_fare = fare_cleaned.median()

  sns.histplot(fare_cleaned, kde=True, ax=axes[index], color='skyblue')
  axes[index].axvline(mean_fare, color='red', linestyle='--', label=f'Mean ({mean_fare:.1f})')
  axes[index].axvline(median_fare, color='green', linestyle='-', label=f'Median ({median_fare:.1f})')
  axes[index].set_title(f'Fare Distribution: \"{passenger_class}\" Pclass')
  axes[index].legend()
  index+=1

Zatem tak samo jak przy wieku, do uzupełnienia danych użyjemy metody medianowej

In [ ]:
titanic_df['Fare'] = titanic_df.groupby(['Pclass'])['Fare'].transform(lambda x: x.fillna(x.median()))
titanic_df['Fare'].isna().sum()

### Miejsce Wejścia Na Pokład (Embarked)

In [ ]:
print(titanic_df['Embarked'].unique())
print(titanic_df[titanic_df['Embarked'].isna()])

To są jedyni pasażerowie z niewiadomą wartością `Embarked`.
Uważam, iż poprzez analizę wartości `Fare` na tle `Embarked` jestem w stanie poprawnie zgadnąć brakujące wartości

In [ ]:
summary_stats = titanic_df.groupby('Embarked')['Fare'].agg(
    ['count', 'mean', 'median']
)
print(summary_stats)

In [ ]:
boxplot(titanic_df[titanic_df['Embarked'] == 'S'], "Fare")

In [ ]:
boxplot(titanic_df[titanic_df['Embarked'] == 'C'], "Fare")

In [ ]:
boxplot(titanic_df[titanic_df['Embarked'] == 'Q'], "Fare")

Co boxploty nam mówią:
* Wiekszość pasażerów z lokalizacji `Q` kupiło bilet poniżej $20                    -> *nie pasuje* :(
* Wiekszość pasażerów z lokalizacji `C` kupiło bilet w granicach $5 - $90           -> *2+dobrze* :>>>
* Wiekszość pasażerów z lokalizacji `S` kupiło bilet w granicach $0 - $35           -> *nie pasuje* :(

A zatem uważam, iż najlepiej jest uzupełnić wartości portem 'C'.

In [ ]:
titanic_df['Embarked'] = titanic_df['Embarked'].fillna('C')

Nie mamy już braków:

In [ ]:
titanic_df.isna().sum()  # Returns the count of NaNs for each column

## Typy Danych

In [ ]:
titanic_df.info()

Jeśli Survive ma być wartością logiczną, to zamiast int64 może być wartością typu bool

In [ ]:
titanic_df['Survived'] = titanic_df['Survived'].astype(bool)
titanic_df.info()

'Pclass', 'Embarked' & 'Location' to wartosci kategoryzujace o skończenie małym zakresie wartości, idealne dla *category* typu danych

In [ ]:
titanic_df['Pclass'] = titanic_df['Pclass'].astype("category")
titanic_df['Location'] = titanic_df['Location'].astype("category")
titanic_df['Embarked'] = titanic_df['Embarked'].astype("category")
titanic_df.info()

Sprawdzę teraz czy wszystkie wartosci column Age lub Fare mozna przecastować na integery

In [ ]:
if (titanic_df['Age'].dropna() % 1 == 0).all():
  print("Można!")
else:
  print("Nie można!")
  print(titanic_df[titanic_df['Age'] % 1 != 0])

In [ ]:
if (titanic_df['Fare'].dropna() % 1 == 0).all():
  print("Można!")
else:
  print("Nie można!")
  print(titanic_df[titanic_df['Fare'] % 1 != 0])

*Jak nie można, to nie można*

Sprawdzę czy da się uszczuplić 'Family'

In [ ]:
print(titanic_df['Family'].min())
print(titanic_df['Family'].max())

In [ ]:
titanic_df['Family'] = titanic_df['Family'].astype("int8")
titanic_df.info()

## Encoding

Trudno doszukiwać się jakieś reguły w wartościach `Embarked`, a ilość możliwych wartości jest wystarczająco mała byśmy użyli one-hot encodingu.

In [ ]:
titanic_df = one_hot_encode_column(titanic_df, 'Embarked')
titanic_df.info()

Location ma dyskretną wartość z małej puli znanyh wartości. Zważając na relację fizyczną lokalizacji (A -> B -> ...) proponuję ordinal encoding.

In [ ]:
mapping = {
    'A' : 0,
    'B' : 1,
    'C' : 2,
    'D' : 3,
    'E' : 4,
    'F' : 5,
    'G' : 6,
    'T' : 7,
    'U' : 8
}
titanic_df['Location'] = titanic_df['Location'].map(mapping)
titanic_df.Location.value_counts()

In [ ]:
mapping = {
    1 : 0,
    2 : 1,
    3 : 2
}
titanic_df['Pclass'] = titanic_df['Pclass'].map(mapping)
titanic_df.Pclass.value_counts()

'Survived' oraz 'IsMale' jest już binary-encoded, Age, Fare i Family to dane numeryczne, więc jak na mnie to git

## Outliery

In [ ]:
boxplot(titanic_df, 'Family')

In [ ]:
boxplot(titanic_df, 'Fare')

Jak widać, istnieją outliery zarówno dla Fare i Family.
Outlierów natury kategorycznej raczej nie ma

## Troche wizualizacji danych

In [ ]:
categorical_and_survived(titanic_df, "Pclass")

In [ ]:
categorical_and_survived(titanic_df, "Location")

In [ ]:
categorical_and_survived(titanic_df, "Family")

In [ ]:
plot_numeric_histogram(titanic_df, 'Fare')

In [ ]:
plot_numeric_histogram(titanic_df, 'Age')